In [1]:
import os
import re
import pandas as pd


FILENAME_PATTERN = re.compile(
    r"^(?P<sample_id>.+?)_(?P<tool>fusioncatcher|arriba)_case_specific_filtered\.tsv$"
)


def parse_filename(filename):
    match = FILENAME_PATTERN.match(filename)
    if match is None:
        return None, None
    return match.group("sample_id"), match.group("tool")


def clean_arriba_fusion_transcript(fusion_transcript):
    if fusion_transcript is None or fusion_transcript.strip() in ("", "."):
        return None, None, None, False

    sequence = fusion_transcript.strip()
    truncated = False

    if "..." in sequence:
        sequence = sequence.split("...")[0]
        truncated = True

    parts = sequence.split("|")

    if len(parts) == 2:
        left_raw, right_raw = parts
        insertion_raw = ""
    elif len(parts) == 3:
        left_raw, insertion_raw, right_raw = parts
    else:
        return None, None, None, truncated

    def clean_segment(segment):
        segment = segment.replace("___", "")
        segment = segment.replace("-", "")
        segment = segment.replace("[", "").replace("]", "")
        segment = segment.replace("?", "N")
        segment = segment.upper()
        return segment

    left_fragment = clean_segment(left_raw)
    right_fragment = clean_segment(right_raw)
    insertion_sequence = clean_segment(insertion_raw)

    return left_fragment, right_fragment, insertion_sequence, truncated


def parse_fusioncatcher_row(row, sample_id):
    dna_fs = row.get("Fusion_sequence", "")
    return {
        "Sample_ID": sample_id,
        "Tool": "fusioncatcher",
        "Gene_A_name": row.get("Gene_1_symbol(5end_fusion_partner)", ""),
        "Gene_B_name": row.get("Gene_2_symbol(3end_fusion_partner)", ""),
        "Gene_A_ensembl_id": row.get("Gene_1_id(5end_fusion_partner)", ""),
        "Gene_B_ensembl_id": row.get("Gene_2_id(3end_fusion_partner)", ""),
        "dna_fs": dna_fs,
        "Insertion_sequence": "",
        "reported_frame": row.get("Predicted_effect", ""),
        "Spanning_unique_reads": row.get("Spanning_unique_reads", ""),
        "Spanning_pairs": row.get("Spanning_pairs", ""),
        "Truncated_due_to_low_coverage": False,
        "Parse_successful": dna_fs not in ("", None) and "*" in dna_fs,
    }


def parse_arriba_row(row, sample_id):
    fusion_transcript = row.get("fusion_transcript", "")
    left_fragment, right_fragment, insertion_sequence, truncated = \
        clean_arriba_fusion_transcript(fusion_transcript)

    parse_successful = left_fragment is not None and right_fragment is not None
    dna_fs = (left_fragment + "*" + right_fragment) if parse_successful else ""

    return {
        "Sample_ID": sample_id,
        "Tool": "arriba",
        "Gene_A_name": row.get("gene1", ""),
        "Gene_B_name": row.get("gene2", ""),
        "Gene_A_ensembl_id": row.get("gene_id1", ""),
        "Gene_B_ensembl_id": row.get("gene_id2", ""),
        "dna_fs": dna_fs,
        "Insertion_sequence": insertion_sequence if insertion_sequence else "",
        "reported_frame": row.get("reading_frame", ""),
        "Confidence": row.get("confidence", ""),
        "Arriba_peptide_sequence": row.get("peptide_sequence", ""),
        "Split_reads1": row.get("split_reads1", ""),
        "Split_reads2": row.get("split_reads2", ""),
        "Truncated_due_to_low_coverage": truncated,
        "Parse_successful": parse_successful,
    }


def load_all_sample_fusions(folder_path):
    all_fusion_records = []

    for filename in sorted(os.listdir(folder_path)):
        sample_id, tool = parse_filename(filename)
        if sample_id is None:
            continue

        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path, sep="\t")
        df.columns = [column.lstrip("#") for column in df.columns]

        for _, row in df.iterrows():
            if tool == "fusioncatcher":
                record = parse_fusioncatcher_row(row, sample_id)
            else:
                record = parse_arriba_row(row, sample_id)
            all_fusion_records.append(record)

    return pd.DataFrame(all_fusion_records)


if __name__ == "__main__":
    folder_path = input("Enter path to the folder containing per-sample fusion TSVs: ")
    master_fusion_table = load_all_sample_fusions(folder_path)
    print("\nTotal fusion records loaded:", len(master_fusion_table))
    master_fusion_table.to_csv("master_fusion_table.csv", index=False)
    print("Saved as master_fusion_table.csv")

Enter path to the folder containing per-sample fusion TSVs:  /home/jerrybryt/MSBT/IP/Gene_Fusions/Case_specific_fusions/Case_specific_fusions



Total fusion records loaded: 1975
Saved as master_fusion_table.csv


In [ ]:
# import time
import requests
import pandas as pd
from Bio import Entrez, SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

Entrez.email = "ojeromebright@gmail.com"


genetic_code = {
    "ATA":"I","ATC":"I","ATT":"I","ATG":"M","ACA":"T","ACC":"T","ACG":"T","ACT":"T",
    "AAC":"N","AAT":"N","AAA":"K","AAG":"K","AGC":"S","AGT":"S","AGA":"R","AGG":"R",
    "CTA":"L","CTC":"L","CTG":"L","CTT":"L","CCA":"P","CCC":"P","CCG":"P","CCT":"P",
    "CAC":"H","CAT":"H","CAA":"Q","CAG":"Q","CGA":"R","CGC":"R","CGG":"R","CGT":"R",
    "GTA":"V","GTC":"V","GTG":"V","GTT":"V","GCA":"A","GCC":"A","GCG":"A","GCT":"A",
    "GAC":"D","GAT":"D","GAA":"E","GAG":"E","GGA":"G","GGC":"G","GGG":"G","GGT":"G",
    "TCA":"S","TCC":"S","TCG":"S","TCT":"S","TTC":"F","TTT":"F","TTA":"L","TTG":"L",
    "TAC":"Y","TAT":"Y","TAA":"*","TAG":"*","TGA":"*"
}

STANDARD_AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWY")
PEPTIDE_LENGTHS = [8, 9, 10, 11]


def translate(dna):
    protein = ""
    for i in range(0, len(dna) - 2, 3):
        codon = dna[i:i + 3]
        amino_acid = genetic_code.get(codon, "X")
        if amino_acid == "*":
            break
        protein += amino_acid
    return protein


# ============================================================
# SOURCE 1: NCBI RefSeq -- multiple candidate transcripts per gene
# ============================================================

def get_cds_candidates_ncbi(gene, max_candidates=5):
    print(f"  [NCBI] searching RefSeq for {gene}")
    time.sleep(0.34)
    candidates = []

    try:
        search = Entrez.esearch(
            db="nucleotide",
            term=f"{gene}[Gene] AND Homo sapiens[Organism] AND biomol_mrna[PROP] AND srcdb_refseq[PROP]"
        )
        result = Entrez.read(search)

        if len(result["IdList"]) == 0:
            print(f"  [NCBI] no RefSeq mRNA found for {gene}")
            return candidates

        for nucleotide_id in result["IdList"]:
            if len(candidates) >= max_candidates:
                break
            time.sleep(0.34)
            handle = Entrez.efetch(db="nucleotide", id=nucleotide_id, rettype="gb", retmode="text")
            record = SeqIO.read(handle, "genbank")

            for feature in record.features:
                if feature.type == "CDS":
                    cds = feature.extract(record.seq)
                    print(f"  [NCBI] candidate: {record.id} for {gene}")
                    candidates.append(str(cds).upper())
                    break

        return candidates

    except Exception as error:
        print(f"  [NCBI] error for {gene}: {error}")
        return candidates


# ============================================================
# SOURCE 2: Ensembl REST -- candidate transcripts matching the SAME
# annotation (GENCODE/Ensembl) that the fusion caller's breakpoint was
# originally called against. NCBI RefSeq is a separately curated
# transcript set and is not guaranteed to contain the same exon
# structure Ensembl used -- this is a likely driver of "fragment not
# found" failures, especially in highly duplicated/paralogous gene
# families (e.g. POM121/POM121C/RBAK).
# ============================================================

ENSEMBL_SERVER = "https://rest.ensembl.org"


def get_cds_candidates_ensembl(ensembl_gene_id, max_candidates=10):
    candidates = []

    if not ensembl_gene_id or not isinstance(ensembl_gene_id, str):
        return candidates

    clean_id = ensembl_gene_id.split(".")[0]

    if not clean_id.startswith("ENSG"):
        return candidates

    try:
        time.sleep(0.1)
        lookup_resp = requests.get(
            f"{ENSEMBL_SERVER}/lookup/id/{clean_id}",
            params={"expand": 1},
            headers={"Content-Type": "application/json"},
            timeout=15,
        )

        if not lookup_resp.ok:
            print(f"  [Ensembl] lookup failed for {clean_id}: HTTP {lookup_resp.status_code}")
            return candidates

        gene_data = lookup_resp.json()
        transcripts = gene_data.get("Transcript", [])

        for transcript in transcripts:
            if len(candidates) >= max_candidates:
                break

            transcript_id = transcript.get("id")
            if transcript_id is None:
                continue

            time.sleep(0.1)
            seq_resp = requests.get(
                f"{ENSEMBL_SERVER}/sequence/id/{transcript_id}",
                params={"type": "cds"},
                headers={"Content-Type": "application/json"},
                timeout=15,
            )

            if not seq_resp.ok:
                continue

            seq_data = seq_resp.json()
            cds_seq = seq_data.get("seq")

            if cds_seq:
                print(f"  [Ensembl] candidate: {transcript_id} for {clean_id}")
                candidates.append(cds_seq.upper())

        return candidates

    except Exception as error:
        print(f"  [Ensembl] error for {clean_id}: {error}")
        return candidates


# ============================================================
# COMBINED: NCBI + Ensembl candidates, cached per gene
# ============================================================

def get_cds_candidates(gene_name, ensembl_id, cds_cache):
    cache_key = (gene_name, ensembl_id)

    if cache_key in cds_cache:
        return cds_cache[cache_key]

    print(f"Retrieving CDS candidates for {gene_name} ({ensembl_id})")

    ncbi_candidates = get_cds_candidates_ncbi(gene_name)
    ensembl_candidates = get_cds_candidates_ensembl(ensembl_id)

    combined = list(dict.fromkeys(ncbi_candidates + ensembl_candidates))

    print(f"  Total candidates for {gene_name}: {len(combined)} "
          f"(NCBI: {len(ncbi_candidates)}, Ensembl: {len(ensembl_candidates)})")

    cds_cache[cache_key] = combined
    return combined


# ============================================================
# PEPTIDE PROPERTY CALCULATION
# ============================================================

def calculate_peptide_properties(peptide):
    if not set(peptide).issubset(STANDARD_AMINO_ACIDS):
        return None

    analysis = ProteinAnalysis(peptide)
    mw = analysis.molecular_weight()
    gravy = analysis.gravy()
    aromaticity = analysis.aromaticity()
    instability = analysis.instability_index()
    pI = analysis.isoelectric_point()

    if hasattr(analysis, "get_amino_acids_percent"):
        aa_percent = analysis.get_amino_acids_percent()
    else:
        aa_percent = analysis.amino_acids_percent

    aa_percent_total = sum(aa_percent.values())
    if aa_percent_total > 0:
        aa_percent = {aa: value / aa_percent_total for aa, value in aa_percent.items()}

    hydrophobic = sum(aa_percent[a] for a in "AVILMFWY")
    polar = sum(aa_percent[a] for a in "STNQC")
    positive = peptide.count("K") + peptide.count("R") + peptide.count("H")
    negative = peptide.count("D") + peptide.count("E")
    net_charge = positive - negative

    return {
        "Molecular_weight": round(mw, 2), "GRAVY": round(gravy, 3),
        "Aromaticity": round(aromaticity, 3), "Instability_index": round(instability, 2),
        "Isoelectric_point": round(pI, 2), "Hydrophobic_fraction": round(hydrophobic, 3),
        "Polar_fraction": round(polar, 3), "Net_charge": net_charge,
    }


def search_human_proteome(peptide, human_proteins):
    for protein_id, sequence in human_proteins:
        if peptide in sequence:
            return protein_id
    return None


# ============================================================
# CORE FUSION PROCESSING
# ============================================================

def process_fusion(dna_fs, gene_a_cds_candidates, gene_b_cds_candidates, reported_frame,
                    gene_a_name, gene_b_name, sample_id, tool, human_proteins):

    stage_counts = {"candidates_generated": 0, "novel_before_proteome": 0}
    dna_fs = dna_fs.replace(" ", "").upper()

    if "*" not in dna_fs:
        return [], "no breakpoint marker in dna_fs", stage_counts, None

    left_fragment, right_fragment = dna_fs.split("*")
    reconstruction = None

    for candidate_gene_a_cds in gene_a_cds_candidates:
        gene_a = candidate_gene_a_cds.replace(" ", "").upper()

        for candidate_gene_b_cds in gene_b_cds_candidates:
            gene_b = candidate_gene_b_cds.replace(" ", "").upper()

            left_in_a = gene_a.find(left_fragment)
            left_in_b = gene_b.find(left_fragment)
            right_in_a = gene_a.find(right_fragment)
            right_in_b = gene_b.find(right_fragment)

            if left_in_a != -1 and right_in_b != -1:
                left_sequence = gene_a[:left_in_a + len(left_fragment)]
                right_sequence = gene_b[right_in_b:]
                upstream_gene_name, downstream_gene_name = "Gene A", "Gene B"
                downstream_index = right_in_b
                reconstruction = (left_sequence, right_sequence, upstream_gene_name,
                                   downstream_gene_name, downstream_index,
                                   candidate_gene_a_cds, candidate_gene_b_cds)
                break
            elif left_in_b != -1 and right_in_a != -1:
                left_sequence = gene_b[:left_in_b + len(left_fragment)]
                right_sequence = gene_a[right_in_a:]
                upstream_gene_name, downstream_gene_name = "Gene B", "Gene A"
                downstream_index = right_in_a
                reconstruction = (left_sequence, right_sequence, upstream_gene_name,
                                   downstream_gene_name, downstream_index,
                                   candidate_gene_a_cds, candidate_gene_b_cds)
                break

        if reconstruction is not None:
            break

    if reconstruction is None:
        return [], "unable to reconstruct fusion CDS (fragments not found in any candidate transcript)", stage_counts, None

    (left_sequence, right_sequence, upstream_gene_name, downstream_gene_name,
     downstream_index, gene_a_cds, gene_b_cds) = reconstruction

    fusion_cds = left_sequence + right_sequence
    junction = len(left_sequence)
    protein = translate(fusion_cds)
    gene_a_protein = translate(gene_a_cds)
    gene_b_protein = translate(gene_b_cds)
    junction_aa = junction // 3

    if junction_aa >= len(protein):
        return [], "junction falls beyond translated protein (early stop codon)", stage_counts, None

    calculated_frame_math = "in-frame" if junction % 3 == downstream_index % 3 else "out-of-frame"

    downstream_native_protein = gene_b_protein if downstream_gene_name == "Gene B" else gene_a_protein
    fusion_downstream_protein = protein[junction_aa:]
    downstream_junction_aa = downstream_index // 3
    native_downstream_protein = downstream_native_protein[downstream_junction_aa:]
    comparison_length = min(30, len(fusion_downstream_protein), len(native_downstream_protein))

    if comparison_length > 0:
        matches = sum(1 for i in range(comparison_length) if fusion_downstream_protein[i] == native_downstream_protein[i])
        identity = matches / comparison_length
    else:
        identity = 0.0

    calculated_frame_identity = "in-frame" if identity >= 0.90 else "out-of-frame"
    frame_methods_agree = (calculated_frame_math == calculated_frame_identity)
    calculated_frame = calculated_frame_math

    if calculated_frame == "in-frame":
        novel_start_aa, novel_end_aa = junction_aa, junction_aa
    else:
        novel_start_aa, novel_end_aa = junction_aa, len(protein) - 1

    novel_peptides = []

    for peptide_length in PEPTIDE_LENGTHS:
        earliest_start = max(0, novel_start_aa - peptide_length + 1)
        latest_start = min(len(protein) - peptide_length, novel_end_aa)
        if latest_start < earliest_start:
            continue

        for start in range(earliest_start, latest_start + 1):
            end = start + peptide_length - 1
            peptide = protein[start:start + peptide_length]

            if start <= junction_aa <= end:
                kind = "junction"
            elif calculated_frame == "out-of-frame" and start > junction_aa:
                kind = "downstream"
            else:
                continue

            stage_counts["candidates_generated"] += 1

            if peptide not in gene_a_protein and peptide not in gene_b_protein:
                novel_peptides.append((start, peptide, kind, peptide_length))

    stage_counts["novel_before_proteome"] = len(novel_peptides)

    if len(novel_peptides) == 0:
        return [], "no novel peptides survived (all matched native gene A/B protein)", stage_counts, None

    peptide_records = []

    for start, peptide, kind, peptide_length in novel_peptides:
        peptide_properties = calculate_peptide_properties(peptide)
        protein_match = search_human_proteome(peptide, human_proteins)
        human_novel = "YES" if protein_match is None else "NO"

        record = {
            "Sample_ID": sample_id, "Tool": tool, "GeneA_name": gene_a_name, "GeneB_name": gene_b_name,
            "Upstream_gene": upstream_gene_name, "Downstream_gene": downstream_gene_name,
            "Reported_frame": reported_frame, "Calculated_frame_math": calculated_frame_math,
            "Calculated_frame_identity": calculated_frame_identity, "Frame_methods_agree": frame_methods_agree,
            "Peptide_type": kind, "Peptide": peptide, "Peptide_length": peptide_length,
            "AA_position": start, "DNA_junction": junction, "AA_junction": junction_aa,
            "Fusion_sequence": dna_fs, "Novel_to_human": human_novel, "Matching_protein": protein_match,
        }

        if peptide_properties is not None:
            record.update(peptide_properties)
        else:
            record["Properties_calculated"] = False

        peptide_records.append(record)

    fusion_protein_info = {
        "Sample_ID": sample_id, "Tool": tool, "GeneA_name": gene_a_name, "GeneB_name": gene_b_name,
        "Upstream_gene": upstream_gene_name, "Downstream_gene": downstream_gene_name,
        "Calculated_frame_math": calculated_frame_math, "Calculated_frame_identity": calculated_frame_identity,
        "DNA_junction": junction, "AA_junction": junction_aa, "Full_fusion_protein": protein,
        "Full_fusion_protein_length": len(protein), "Fusion_sequence": dna_fs,
    }

    return peptide_records, None, stage_counts, fusion_protein_info


# ============================================================
# MAIN PIPELINE
# ============================================================

if __name__ == "__main__":

    aggregate_db_path = input("Enter path to all_fusions_database.tsv: ")
    sample_folder_path = input("Enter path to the folder of per-sample fusion TSVs: ")
    proteome_fasta_path = input("Enter path to human_sprot.fasta: ")

    counts = {
        "aggregate_rows_total": 0, "aggregate_rows_qualifying": 0, "unique_qualifying_pairs": 0,
        "master_table_rows_total": 0, "rows_after_join": 0, "fusions_attempted": 0,
        "failed_cds_retrieval": 0, "failed_reconstruction": 0, "failed_early_stop": 0,
        "failed_no_novel_peptides": 0, "fusions_succeeded": 0, "total_candidate_peptides": 0,
        "total_novel_peptides_before_proteome": 0, "total_final_peptide_records": 0,
        "final_peptides_novel_to_proteome": 0,
    }

    agg_df = pd.read_csv(aggregate_db_path, sep="\t")
    counts["aggregate_rows_total"] = len(agg_df)
    agg_df = agg_df[agg_df["frequency"] >= 3]
    counts["aggregate_rows_qualifying"] = len(agg_df)

    qualifying_pairs = set()
    for _, row in agg_df.iterrows():
        parts = str(row["canonical_pair"]).split("|")
        if len(parts) == 2:
            qualifying_pairs.add(frozenset([parts[0].strip(), parts[1].strip()]))
    counts["unique_qualifying_pairs"] = len(qualifying_pairs)

    master_fusion_table = load_all_sample_fusions(sample_folder_path)
    counts["master_table_rows_total"] = len(master_fusion_table)

    def pair_qualifies(row):
        return frozenset([row["Gene_A_name"], row["Gene_B_name"]]) in qualifying_pairs

    mask = master_fusion_table.apply(pair_qualifies, axis=1) & master_fusion_table["Parse_successful"]
    filtered_table = master_fusion_table[mask].reset_index(drop=True)
    counts["rows_after_join"] = len(filtered_table)

    print("Loading Swiss-Prot human proteome...")
    human_proteins = [(record.id, str(record.seq)) for record in SeqIO.parse(proteome_fasta_path, "fasta")]
    print("Proteins loaded:", len(human_proteins))

    cds_cache = {}
    all_results = []
    all_fusion_proteins = []
    failures = []

    for i, row in filtered_table.iterrows():
        counts["fusions_attempted"] += 1
        print(f"\n[{i + 1}/{len(filtered_table)}] {row['Sample_ID']} | {row['Tool']} | "
              f"{row['Gene_A_name']}-{row['Gene_B_name']}")

        gene_a_cds_candidates = get_cds_candidates(row["Gene_A_name"], row.get("Gene_A_ensembl_id", ""), cds_cache)
        gene_b_cds_candidates = get_cds_candidates(row["Gene_B_name"], row.get("Gene_B_ensembl_id", ""), cds_cache)

        if len(gene_a_cds_candidates) == 0 or len(gene_b_cds_candidates) == 0:
            counts["failed_cds_retrieval"] += 1
            failures.append({"Sample_ID": row["Sample_ID"], "Tool": row["Tool"],
                              "Gene_A_name": row["Gene_A_name"], "Gene_B_name": row["Gene_B_name"],
                              "Reason": "CDS retrieval failed for one or both genes"})
            continue

        peptide_records, failure_reason, stage_counts, fusion_protein_info = process_fusion(
            dna_fs=row["dna_fs"], gene_a_cds_candidates=gene_a_cds_candidates,
            gene_b_cds_candidates=gene_b_cds_candidates, reported_frame=row["reported_frame"],
            gene_a_name=row["Gene_A_name"], gene_b_name=row["Gene_B_name"],
            sample_id=row["Sample_ID"], tool=row["Tool"], human_proteins=human_proteins,
        )

        counts["total_candidate_peptides"] += stage_counts["candidates_generated"]
        counts["total_novel_peptides_before_proteome"] += stage_counts["novel_before_proteome"]

        if failure_reason is not None:
            if "reconstruct" in failure_reason:
                counts["failed_reconstruction"] += 1
            elif "early stop" in failure_reason:
                counts["failed_early_stop"] += 1
            elif "no novel peptides" in failure_reason:
                counts["failed_no_novel_peptides"] += 1
            failures.append({"Sample_ID": row["Sample_ID"], "Tool": row["Tool"],
                              "Gene_A_name": row["Gene_A_name"], "Gene_B_name": row["Gene_B_name"],
                              "Reason": failure_reason})
            continue

        counts["fusions_succeeded"] += 1
        all_results.extend(peptide_records)
        if fusion_protein_info is not None:
            all_fusion_proteins.append(fusion_protein_info)

    results_df = pd.DataFrame(all_results)
    results_df.to_csv("Final_candidate_peptides.csv", index=False)

    fusion_proteins_df = pd.DataFrame(all_fusion_proteins)
    fusion_proteins_df.to_csv("Full_fusion_proteins.csv", index=False)

    counts["total_final_peptide_records"] = len(results_df)
    if len(results_df) > 0 and "Novel_to_human" in results_df.columns:
        counts["final_peptides_novel_to_proteome"] = int((results_df["Novel_to_human"] == "YES").sum())

    failures_df = pd.DataFrame(failures)
    failures_df.to_csv("Failed_fusions_log.csv", index=False)

    counts_df = pd.DataFrame(list(counts.items()), columns=["Stage", "Count"])
    counts_df.to_csv("Pipeline_stage_counts.csv", index=False)

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE -- STAGE-BY-STAGE COUNTS")
    print("=" * 60)
    for stage, count in counts.items():
        print(f"{stage}: {count}")
    print("\nSaved: Final_candidate_peptides.csv, Full_fusion_proteins.csv, Failed_fusions_log.csv, Pipeline_stage_counts.csv")